# Monitoring Dashboard - Pipeline Health

**Metrics tracked:**
- Processing latency
- Error rate
- Data quality (missing, duplicates)
- System resources (CPU, RAM)
- Model accuracy

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Config
DB_PATH = Path('data/bigdata.db')
METRICS_DIR = Path('monitoring/metrics')
ALERTS_DIR = Path('monitoring/alerts')
METRICS_DIR.mkdir(parents=True, exist_ok=True)
ALERTS_DIR.mkdir(parents=True, exist_ok=True)

# SQLite connection
def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

## 2. Database Status

In [2]:
# Check database exists
if DB_PATH.exists():
    conn = get_db_connection()
    
    # List tables
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", 
        conn
    )
    
    print('📁 DATABASE STATUS')
    print(f'   Location: {DB_PATH}')
    print(f'   Size: {DB_PATH.stat().st_size / (1024*1024):.2f} MB')
    print(f'\n📋 Tables:')
    for table in tables['name']:
        count = pd.read_sql_query(f'SELECT COUNT(*) as c FROM {table}', conn)['c'][0]
        print(f'   - {table}: {count:,} rows')
    
    conn.close()
else:
    print('❌ Database not found. Run ETL_Pipeline.ipynb first!')

📁 DATABASE STATUS
   Location: data\bigdata.db
   Size: 7082.44 MB

📋 Tables:


   - events: 38,254,680 rows
   - sqlite_sequence: 1 rows
   - pipeline_metadata: 0 rows
   - user_clusters: 100,000 rows


   - churn_predictions: 1,590,107 rows
   - product_recommendations: 60 rows
   - model_registry: 2 rows


## 3. Pipeline Metadata

In [3]:
if DB_PATH.exists():
    conn = get_db_connection()
    
    # Pipeline metadata
    metadata = pd.read_sql_query('SELECT * FROM pipeline_metadata ORDER BY processed_at DESC', conn)
    
    if len(metadata) > 0:
        print('📊 PIPELINE HISTORY')
        display(metadata)
    else:
        print('No pipeline runs recorded.')
    
    conn.close()
else:
    print('Database not available')

No pipeline runs recorded.


## 4. Data Quality Checks

In [4]:
if DB_PATH.exists():
    conn = get_db_connection()
    
    # Load sample data
    df = pd.read_sql_query('SELECT * FROM events LIMIT 10000', conn)
    
    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'column': missing.index, 
        'missing_count': missing.values, 
        'missing_pct': missing_pct.values
    })
    missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    
    print('📊 DATA QUALITY - Missing Values')
    if len(missing_df) > 0:
        display(missing_df)
    else:
        print('No missing values found!')
    
    # Duplicate check
    dup_count = df.duplicated().sum()
    dup_pct = dup_count / len(df) * 100
    print(f'\n📊 DATA QUALITY - Duplicates')
    print(f'Duplicate rows: {dup_count:,} ({dup_pct:.2f}%)')
    
    conn.close()
else:
    print('Database not available')

📊 DATA QUALITY - Missing Values
No missing values found!

📊 DATA QUALITY - Duplicates
Duplicate rows: 0 (0.00%)


## 5. Model Performance

In [5]:
registry_file = Path('models/model_registry.json')

if registry_file.exists():
    with open(registry_file) as f:
        registry = json.load(f)
    
    print('🤖 TRAINED MODELS')
    for model in registry['models']:
        print(f'\n  {model["name"]}')
        print(f'    Type: {model["type"]}')
        print(f'    Algorithm: {model["algorithm"]}')
        if 'metrics' in model:
            for metric, value in model['metrics'].items():
                print(f'    {metric}: {value:.4f}')
else:
    print('No models found. Run ML pipeline first!')

🤖 TRAINED MODELS

  customer_segmentation_kmeans
    Type: clustering
    Algorithm: KMeans
    silhouette_score: 0.7067

  churn_prediction_rf
    Type: classification
    Algorithm: RandomForestClassifier
    accuracy: 0.7112
    f1: 0.8129


## 6. Summary Dashboard

In [6]:
print('\n' + '=' * 60)
print('📊 PIPELINE MONITORING DASHBOARD')
print('=' * 60)

print(f'\n⏰ Generated at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

print(f'\n💾 DATABASE')
if DB_PATH.exists():
    print(f'   Status: ✅ Connected')
    print(f'   Location: {DB_PATH}')
    conn = get_db_connection()
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
    for table in tables['name']:
        count = pd.read_sql_query(f'SELECT COUNT(*) as c FROM {table}', conn)['c'][0]
        print(f'   - {table}: {count:,}')
    conn.close()
else:
    print('   Status: ❌ Not found')

print(f'\n🤖 ML MODELS')
if registry_file.exists():
    print(f'   Total models: {len(registry["models"])}')
else:
    print('   No models')

print('\n' + '=' * 60)


📊 PIPELINE MONITORING DASHBOARD

⏰ Generated at: 2026-10-04 22:18:16

💾 DATABASE
   Status: ✅ Connected
   Location: data\bigdata.db


   - events: 38,254,680
   - sqlite_sequence: 1
   - pipeline_metadata: 0
   - user_clusters: 100,000
   - churn_predictions: 1,590,107
   - product_recommendations: 60
   - model_registry: 2

🤖 ML MODELS
   Total models: 2

